# Задание

1. Найти датасет на hugging face, который будет содержать в себе числовые данные (> 3 числовых признаков)

2. Очистить данные от выбросов методами, рассмотренными на практическом занятии

3. Произвести отбор признаков одним из способов, рассмотренных на лекционной части занятия

# Imports

In [1]:
%pip install pandas numpy plotly nbformat scikit-learn datasets


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from datasets import load_dataset

/Users/vsevolodpanteleev/Projects/dvfu/semester-7/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset

In [3]:
dataset = load_dataset("Darkester/SimpleLinearRegression")
df = dataset["train"].to_pandas()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nDescriptive statistics:\n{df.describe()}")

Dataset shape: (549, 7)
Columns: ['feature1', 'feature2', 'feature3', 'feature4', 'feature5', 'feature6', 'target']

First rows:
   feature1  feature2  feature3  feature4  feature5  feature6  target
0       5.2       3.1       7.8       2.5       4.0       6.3    38.7
1       6.0       2.8       8.2       3.0       3.5       5.9    39.1
2       4.8       3.5       7.5       2.8       4.2       6.1    37.3
3       5.5       3.0       8.0       2.7       3.8       6.0    38.5
4       6.2       2.9       7.9       3.1       3.7       5.8    39.3

Data types:
feature1    float64
feature2    float64
feature3    float64
feature4    float64
feature5    float64
feature6    float64
target      float64
dtype: object

Descriptive statistics:
           feature1      feature2      feature3      feature4      feature5  \
count  5.490000e+02  5.490000e+02  5.490000e+02  5.490000e+02  5.490000e+02   
mean   2.936597e+05  3.694217e+05  6.252158e+05  4.025686e+05  4.972993e+05   
std    9.586300e+05  1

# Outliers

In [4]:
df_cleaned = df.copy()

# Calculate outlier bounds for all columns first
outlier_bounds = {}
for col in df.columns[:-1]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_bounds[col] = (lower, upper)
    
    # Remove outliers from cleaned dataset
    df_cleaned = df_cleaned[(df_cleaned[col] >= lower) & (df_cleaned[col] <= upper)]

# Create subplots for before/after boxplots
fig = make_subplots(
    rows=len(df.columns) - 1, cols=2,
    subplot_titles=[f"{col} - Before" if i % 2 == 0 else f"{col} - After (IQR)" 
                    for i in range(2 * (len(df.columns) - 1))]
)

for idx, col in enumerate(df.columns[:-1]):
    # Before boxplot (using original data)
    fig.add_trace(
        go.Box(y=df[col], name=col, showlegend=False),
        row=idx + 1, col=1
    )
    
    # After boxplot (using cleaned data)
    fig.add_trace(
        go.Box(y=df_cleaned[col], name=col, showlegend=False),
        row=idx + 1, col=2
    )
    
    print(f"=== {col} ===")
    print(f"Before: count={df[col].count()}, mean={df[col].mean():.2f}, std={df[col].std():.2f}")
    print(f"After: count={df_cleaned[col].count()}, mean={df_cleaned[col].mean():.2f}, std={df_cleaned[col].std():.2f}\n")

fig.update_layout(height=300 * (len(df.columns) - 1), title_text="Outlier Detection: Before and After IQR Cleaning")
fig.show()

print(f"Rows removed: {len(df) - len(df_cleaned)} ({(1 - len(df_cleaned)/len(df))*100:.1f}%)")

=== feature1 ===
Before: count=549, mean=293659.74, std=958630.04
After: count=390, mean=18.48, std=20.36

=== feature2 ===
Before: count=549, mean=369421.74, std=1195197.69
After: count=390, mean=15.48, std=16.34

=== feature3 ===
Before: count=549, mean=625215.82, std=1958389.57
After: count=390, mean=32.81, std=43.78

=== feature4 ===
Before: count=549, mean=402568.65, std=1268374.61
After: count=390, mean=12.02, std=11.20

=== feature5 ===
Before: count=549, mean=497299.25, std=1556273.33
After: count=390, mean=18.05, std=17.23

=== feature6 ===
Before: count=549, mean=717143.24, std=2264902.94
After: count=390, mean=18.56, std=18.05



Rows removed: 159 (29.0%)


# Feature Selection

Там в лекции упомянулось по forward/backward elimination, поэтому сделаю их. Так как задача просто отобрать фичи, не делил еще на валид выборку)

In [5]:
X = df_cleaned.iloc[:, :-1]
y = df_cleaned.iloc[:, -1]

print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
print(f"Target variable: {y.name}")
print(f"\nTarget statistics:\n{y.describe()}")

Features: 6, Samples: 390
Target variable: target

Target statistics:
count     390.000000
mean      164.967974
std       364.150967
min         0.070000
25%        37.125000
50%        67.300000
75%       100.500000
max      2250.000000
Name: target, dtype: float64


In [6]:
# Method 1: SelectKBest with f_regression
selector = SelectKBest(score_func=f_regression, k=min(5, X.shape[1]))
X_selected = selector.fit_transform(X, y)

scores = pd.DataFrame({'Feature': X.columns, 'Score': selector.scores_}).sort_values('Score', ascending=False)
print("=== SelectKBest (F-regression) ===")
print(scores)
top_features_kbest = scores['Feature'].head(5).tolist()

# Method 2: Random Forest feature importance
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

importance = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print("\n=== Random Forest Importance ===")
print(importance)
top_features_rf = importance['Feature'].head(5).tolist()

# Method 3: Correlation with target
correlation = X.corrwith(y).abs().sort_values(ascending=False)
print("\n=== Correlation with Target ===")
print(correlation)
top_features_corr = correlation.head(5).index.tolist()

=== SelectKBest (F-regression) ===
    Feature        Score
2  feature3  1573.958867
0  feature1   399.172160
4  feature5   128.378832
1  feature2    27.346098
5  feature6     9.687835
3  feature4     0.149843

=== Random Forest Importance ===
    Feature  Importance
2  feature3    0.898711
5  feature6    0.042964
0  feature1    0.019365
3  feature4    0.018977
1  feature2    0.010033
4  feature5    0.009950

=== Correlation with Target ===
feature3    0.895678
feature1    0.712107
feature5    0.498612
feature2    0.256592
feature6    0.156078
feature4    0.019648
dtype: float64


In [7]:
# Visualize feature selection results with Plotly
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("SelectKBest (F-regression)", "Random Forest Importance", "Correlation with Target")
)

# Plot 1: SelectKBest
fig.add_trace(
    go.Bar(y=scores.head(10)['Feature'], x=scores.head(10)['Score'], 
           orientation='h', name='F-Score', marker_color='steelblue'),
    row=1, col=1
)

# Plot 2: Random Forest
fig.add_trace(
    go.Bar(y=importance.head(10)['Feature'], x=importance.head(10)['Importance'], 
           orientation='h', name='Importance', marker_color='green'),
    row=1, col=2
)

# Plot 3: Correlation
fig.add_trace(
    go.Bar(y=correlation.head(10).index, x=correlation.head(10).values, 
           orientation='h', name='Correlation', marker_color='coral'),
    row=1, col=3
)

fig.update_xaxes(title_text="Score", row=1, col=1)
fig.update_xaxes(title_text="Importance", row=1, col=2)
fig.update_xaxes(title_text="Absolute Correlation", row=1, col=3)

fig.update_layout(height=500, title_text="Feature Selection Methods Comparison", showlegend=False)
fig.show()

# Find consensus features
all_top = set(top_features_kbest + top_features_rf + top_features_corr)
print(f"\nConsensus features: {all_top}")

# Select features appearing in multiple methods
final_features = list(set([f for f in all_top if 
    sum([f in top_features_kbest, f in top_features_rf, f in top_features_corr]) >= 2]))
print(f"Final selected features: {sorted(final_features)}")

df_final = df_cleaned[final_features + [y.name]]
print(f"\nFinal dataset shape: {df_final.shape}")

# Create interactive table
fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(df_final.columns)),
    cells=dict(values=[df_final[col] for col in df_final.columns])
)])
fig_table.update_layout(title_text="Final Dataset (First 10 rows)")
fig_table.show()


Consensus features: {'feature2', 'feature6', 'feature5', 'feature3', 'feature1', 'feature4'}
Final selected features: ['feature1', 'feature2', 'feature3', 'feature5', 'feature6']

Final dataset shape: (390, 6)
